# Intro to LangChain 1.x: Architecture & Prompt Engineering Foundations

This notebook uses stable **LangChain 1.x** architecture.

LangChain 1.x focuses on explicit, predictable orchestration via **LCEL (LangChain Expression Language)**, unified model initializations, and structural message objects instead of outdated raw string parsing.

**What is Langchain?**

LangChain is an open-source framework that makes it easier to build applications using Large Language Models (LLMs). It acts as the connective layer—or "plumbing"—between an LLM and external tools, databases, and APIs, allowing developers to create advanced, context-aware AI agents rather than just standalone chat prompts.

## 1. Installation Required

We will install the following packages and provider extensions in our virtual environment.

- **getpass** (`pip install getpass4`)
- **langchain** (`pip install langchain`)
- **langchain-core** (`pip install langchain-core`)
- **langchain-groq** (`pip install langchain-groq`)
- **langchain-openai** (`pip install langchain-openai`)
- **langchain-anthropic** (`pip install langchain-anthropic`)

## 2. Topics Covered in This Notebook

To give you a thorough understanding of LangChain 1.x architectures, we will sequentially cover the following concepts:

1. **Environment Setup & Vendor Authentication**
   - Package optimization (`langchain-core` vs. community extensions).
   - Secure API key lifecycle management.

2. **The LangChain Core Foundations**
   - The vendor-agnostic `init_chat_model()` initialization factory.
   - Message Schemas: Understanding structural payload blocks (`SystemMessage`, `HumanMessage`, `AIMessage`).
   - Unified component invocation using `.invoke()`.

3. **Prompt Engineering Foundations**
   - **Zero-Shot Prompting:** Separating roles and structuring system boundaries.
   - **Prompt Templates:** Dynamically and safely parsing variables into explicit payloads.

4. **LangChain Expression Language (LCEL)**
   - Composing explicit pipelines using the pipe (`|`) operator.
   - Isolating clean text returns via standard structural `StrOutputParser` middleware.

5. **Few-Shot Prompting (In-Context Learning)**
   - Constructing structural training demonstration sets.
   - Wrapping arrays cleanly inside the modern `FewShotChatMessagePromptTemplate` engine.

6. **Chaining Multiple Prompts & Graph Parallelization**
   - Creating composite, sequential execution pipelines.
   - Mapping upstream evaluation nodes concurrently and routing structural keys into downstream templates using `RunnablePassthrough`.

7. **Multi-Turn Conversation Formatting (Chat History)**
   - Maintaining stateful contextual awareness between user and model.
   - Utilizing `MessagesPlaceholder` to dynamically inject an growing list of chat objects into prompt schemas natively.

In [1]:
import os
from getpass import getpass  # to enter the password securely

In [2]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# OPTIONAL: Uncomment to use alternative providers
# if "OPENAI_API_KEY" not in os.environ:
#     os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")
    
# if "ANTHROPIC_API_KEY" not in os.environ:
#     os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API Key: ")

## 3. Core Concept: Model Architecture & Swapping Integration

In LangChain 1.x, instead of importing disparate classes like `ChatGroq` or `ChatOpenAI`, the standard best practice is using the unified **`init_chat_model()`** initialization strategy. This allows you to toggle models dynamically based on a simple string descriptor.

Let's initialize our primary engine (Groq) alongside examples for OpenAI and Anthropic.

In [3]:
from langchain.chat_models import init_chat_model

# 1. Primary Setup: Using Groq (Llama 3.3)
model = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq",
    temperature=0.2
)

# =====================================================================
# ALTERNATIVE PROVIDERS (Commented out - uncomment if keys are provided)
# =====================================================================

# 2. OpenAI Configuration Example
# model = init_chat_model(
#     model="gpt-4o",
#     model_provider="openai",
#     temperature=0.2
# )

# 3. Anthropic Claude Configuration Example
# model = init_chat_model(
#     model="claude-3-5-sonnet-latest",
#     model_provider="anthropic",
#     temperature=0.2
# )

print(f"Successfully initialized structural wrapper for provider strategy: {model}")

Successfully initialized structural wrapper for provider strategy: metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}} output_version=None profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x000001C136D80770> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C137675820> model_name='llama-3.3-70b-versatile' temperature=0.2 model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


## 4. Direct Invocation & Message Schemas

Before engineering prompts, we must understand how LangChain structurally manages chat payloads. Messages are represented as typed objects under `langchain.messages` (re-exported from `langchain-core`):

* `SystemMessage`: Sets structural guidelines, persona restrictions, or boundaries.
* `HumanMessage`: Represents user input.
* `AIMessage`: Holds the response structural wrapper generated by the model.

We invoke components natively using the predictable `.invoke()` method.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a history instructor who speaks strictly in concise bullet points."),
    HumanMessage(content="Explain why the roman empire fell.")
]

# Run invocation
response = model.invoke(messages)

print("--- RAW AI MESSAGE OBJECT ---")
print(response)
print("\n--- EXTRACTED CONTENT ---")
print(response.content)

--- RAW AI MESSAGE OBJECT ---
content='* Internal corruption and mismanagement\n* External pressures from barbarian tribes\n* Economic troubles and inflation\n* Military overextension and costly wars\n* Division of the empire into Eastern and Western halves\n* Decline of Roman values and civic engagement\n* Environmental factors like climate change and disease\n* Overreliance on slave labor and decline of the Roman legions' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 56, 'total_tokens': 131, 'completion_time': 0.288610291, 'completion_tokens_details': None, 'prompt_time': 0.00271819, 'prompt_tokens_details': None, 'queue_time': 0.04944932, 'total_time': 0.291328481}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ee391-f0f2-7b83-8d85-176d71d11214-0' tool_calls=[] invalid_tool_calls=[] usag

In [5]:
type(response)  # <class 'langchain.schema.messages.AIMessage'>

langchain_core.messages.ai.AIMessage

## 5. Prompt Engineering Foundations using LangChain

Hardcoding schemas and arrays of message structures becomes brittle when scaled. LangChain offers robust layout templates to decouple design architecture from raw processing.

### A. Zero-Shot Prompt Templates
The `ChatPromptTemplate` constructs an internal structure mapping system states and user injection variables securely.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define a Zero-Shot Prompt layout structure
zero_shot_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert culinary critic. Provide a concise, 2-sentence analytical review of the following dish."),
    ("human", "Analyze this dish: {dish_name}. Key ingredients include: {ingredients}.")
])

# Let's inspect how the prompt dynamically binds parameters
formatted_messages = zero_shot_template.format_messages(
    dish_name="Truffle Cacio e Pepe", 
    ingredients="Pecorino Romano, black pepper, fresh handmade tonnarelli pasta, white truffle oil"
)

print("Formatted Message Payload structure parsed to LLM:")
for msg in formatted_messages:
    print(f"[{type(msg).__name__}]: {msg.content}")

Formatted Message Payload structure parsed to LLM:
[SystemMessage]: You are an expert culinary critic. Provide a concise, 2-sentence analytical review of the following dish.
[HumanMessage]: Analyze this dish: Truffle Cacio e Pepe. Key ingredients include: Pecorino Romano, black pepper, fresh handmade tonnarelli pasta, white truffle oil.


In [8]:
zero_shot_response = model.invoke(formatted_messages)
print("\n--- AI Response to Zero-Shot Prompt ---")
print(zero_shot_response.content)


--- AI Response to Zero-Shot Prompt ---
The Truffle Cacio e Pepe is a masterful iteration of the classic Italian dish, with the rich, umami flavor of Pecorino Romano and the subtle, earthy essence of white truffle oil perfectly balanced by the sharp, peppery kick of black pepper and the tender, handmade tonnarelli pasta. The use of high-quality ingredients and restrained application of truffle oil elevate this comforting, creamy pasta dish to new heights, showcasing a deep understanding of the nuances of Italian cuisine and the art of subtle, sophisticated flavor combination.


### B. Composing execution paths with LCEL (LangChain Expression Language)

Instead of passing templates through boilerplate runners, LangChain uses the unix pipe operator (`|`).

`Pipeline = PromptTemplate | Model | OutputParser`

This establishes an optimized execution pipeline. Let's bind our prompt to our engine and pipe it directly to a `StrOutputParser` to isolate raw text.

In [9]:
# Build the LCEL Chain pipeline
review_chain = zero_shot_template | model | StrOutputParser()

# Invoke the orchestrated chain end-to-end
output = review_chain.invoke({
    "dish_name": "Deconstructed Matcha Cheesecake",
    "ingredients": "Ceremonial grade matcha, white chocolate ganache, black sesame crumble, yuzu gel"
})

print("Pipeline Output:")
print(output)

Pipeline Output:
The Deconstructed Matcha Cheesecake is a masterful reinterpretation of traditional dessert, with the ceremonial grade matcha providing a deep, nuanced bitterness that is beautifully balanced by the creamy white chocolate ganache and the subtle citrus notes of the yuzu gel. The addition of the black sesame crumble adds a satisfying textural element, tying the dish together and elevating it to a harmonious union of Japanese-inspired flavors and modern culinary technique.


### C. Few-Shot Prompting (In-Context Learning)

For structured, complex domain transformations (such as mapping sentiment classifications or specific business logic formatting), instructing the model is rarely enough. Providing few-shot explicit input-output demonstration sets significantly increases reliable outputs.

In [10]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Define explicit operational examples
examples = [
    {"input": "The shipping took two weeks and arrived broken.", "output": "CRITICAL | Logistics | Damaged_Goods"},
    {"input": "I love the sleek design, but the battery life drains in under an hour.", "output": "WARNING | Hardware | Battery_Performance"},
    {"input": "Customer service was amazing and solved my invoice query immediately.", "output": "INFO | Support | Billing_Resolution"}
]

# 2. Package the design schema showing how each example should be formatted
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# 3. Feed examples into the FewShot assembly component
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 4. Combine everything: System Instructions + Few-Shot Examples + Current User Input
final_few_shot_template = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are an automated support ticket tagger. Classify the user's review "
        "matching the exact syntax and classification format shown in the examples below."
    ),
    few_shot_prompt,  # This dynamically injects the human/ai pairs here
    ("human", "{input}")
])

# 5. Compile the final LCEL classification pipeline
classifier_chain = final_few_shot_template | model | StrOutputParser()

# Test with an unseen corporate review
test_review = "The application functions perfectly, but it lacks dark mode configuration options."
classification_result = classifier_chain.invoke({"input": test_review})

print("Few-Shot In-Context Classification Output:")
print(classification_result)

Few-Shot In-Context Classification Output:
MINOR | Software | Feature_Request


## 6. Chaining Multiple Prompts & Parallel Execution (LCEL)

In real-world LLM orchestration, you rarely execute a single prompt in isolation. Often, you need to extract multiple dimensions of meta-data from a user input (e.g., Sentiment and Category) and merge them downstream to generate a final customized action (e.g., a customer support response).

In LangChain 1.x, we achieve this by combining explicit `ChatPromptTemplate` structures with dictionary mapping pipelines. This automatically executes independent upstream chains in parallel for maximum efficiency.

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ==========================================
# CHAIN 1: Sentiment Extraction
# ==========================================
sentiment_template = ChatPromptTemplate.from_messages([
    ("system", "You are an analytical assistant. Determine if the customer review is Positive, Negative, or Neutral. Reply with JUST the single word."),
    ("human", "Review: {review_text}")
])
sentiment_chain = sentiment_template | model | StrOutputParser()

# ==========================================
# CHAIN 2: Category Classification
# ==========================================
category_template = ChatPromptTemplate.from_messages([
    ("system", "Classify the review into exactly one category: Food Quality, Overall Hygiene, Restaurant Ambience, or Customer Service. Reply with JUST the category name."),
    ("human", "Review: {review_text}")
])
category_chain = category_template | model | StrOutputParser()

# ==========================================
# CHAIN 3: Downstream Response Generation
# ==========================================
response_template = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are a professional Guest Relations Manager. Draft a tailored customer engagement response.\n"
        "Guidelines:\n"
        "- If sentiment is Negative, apologize directly for the specific issue category mentioned.\n"
        "- If sentiment is Positive, thank them warmly for highlighting that specific category."
    ),
    (
        "human", 
        "Customer Review: {review_text}\n"
        "Analyzed Sentiment: {sentiment}\n"
        "Identified Category: {category}"
    )
])
final_response_chain = response_template | model | StrOutputParser()

# ==========================================
# COMPLETE EXECUTION GRAPH (LCEL Parallelization)
# ==========================================
# We map the initial user input 'review_text' to run concurrently into both chains,
# then feed all compiled elements into the final response chain.
complete_chain = (
    {
        "review_text": RunnablePassthrough(), # Passes the raw string directly downstream
        "sentiment": sentiment_chain,         # Executes Sentiment Prompt + Model
        "category": category_chain            # Executes Category Prompt + Model
    }
    | final_response_chain                     # Feeds combined dict into Response Prompt + Model
)

# Test Dataset from original notebook
reviews = [
    "The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.",
    "Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit."
]

# Run the complete orchestrator
print("--- TEST 1 (Positive Mix) ---")
print(complete_chain.invoke({"review_text": reviews[0]}))

print("\n" + "="*60 + "\n")

print("--- TEST 2 (Negative Mix) ---")
print(complete_chain.invoke({"review_text": reviews[1]}))

--- TEST 1 (Positive Mix) ---
Dear valued customer,

We are absolutely delighted to hear that you enjoyed your dining experience with us. Thank you warmly for highlighting our chic décor and the ambiance of our restaurant - we're thrilled that you loved the atmosphere we've created. Our team works hard to ensure that every aspect of your visit is exceptional, and it's wonderful to know that our efforts have paid off. We're also glad you enjoyed the seafood platter, and we'll keep striving to serve the freshest and most flavorful dishes.

We appreciate your feedback and look forward to welcoming you back to our restaurant soon. If there's anything we can do to make your next visit even more special, please don't hesitate to let us know.

Best regards,
[Your Name]
Guest Relations Manager


--- TEST 2 (Negative Mix) ---
Dear valued customer,

I am deeply sorry for the disappointing experience you had during your recent visit to our establishment. Specifically, I apologize for the lengthy 

## 7. Multi-Turn Conversation Formatting (Managing Chat History)

In a real-world chat application, the model needs to remember what was said in previous turns. In LangChain 1.x, we do not append raw text to a string. Instead, we maintain a sequence of message objects and inject them dynamically using a **`MessagesPlaceholder`**.

This ensures that the chat history preserves structural roles (`HumanMessage`, `AIMessage`) and is formatted natively for modern chat APIs.

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# 1. Define a template that leaves a slot open for an array of past messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, witty IT support assistant. Keep your answers brief and punchy."),
    # The placeholder dynamically injects an arbitrary list of past turns
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Assemble our basic LCEL pipeline
chat_chain = chat_prompt | model | StrOutputParser()

# 2. Simulate a multi-turn conversation state manually
# In a real app, this list is often retrieved from a database or a memory utility
session_history = [
    HumanMessage(content="Hi, my work laptop is running incredibly slow today."),
    AIMessage(content="Uh oh. Have you tried turning it off and on again? Classic, but it works 80% of the time."),
    HumanMessage(content="Yes, I did. It didn't help. It seems to lag when I open Chrome."),
    AIMessage(content="Ah, Chrome—the ultimate RAM consumer. Let's check your active extensions. Try disabling them.")
]

# 3. Execute the next turn of the conversation
# We pass the historical array alongside the new user question
next_user_query = "I disabled them, and it feels faster now! What should I do next to make sure it stays fast?"

response = chat_chain.invoke({
    "chat_history": session_history,
    "input": next_user_query
})

print("--- NEW AI RESPONSE (With Context Awareness) ---")
print(response)

--- NEW AI RESPONSE (With Context Awareness) ---
Nice fix. Now, update Chrome, and consider clearing browsing data (cookies, cache, etc.). Regular disk cleanups and Windows updates will also keep your laptop zipping along.


## Summary of Best Practices in LangChain 1.x
1. **Always use `.invoke()`** instead of treating components like legacy functions.
2. Initialize models with **`init_chat_model()`** for easier management and cross-vendor standardizations.
3. Build logic via **LCEL (`|`)** chains to maintain visibility, structural modularity, and automatic streaming integrations out of the box.